A notebook for training a deeplabcut model on a directory of videos.

The training process involves initially training and then re-training on outlier cases until the model performs adequately. Here is the typical workflow:

0. divide the videos into directories of training and testing videos because we want to assess the model on untrained videos. 
1. Create or load a .yaml config file pointing to all of the videos of interest.
2. Train:
    1. initial training:
        - extract frames
        - label using the napari GUI
        - check the labels
        - train and test:
            - create training dataset
            - train network
            - evaluate network
            - analyze and automatically label unlabeled videos
    2. refining:
        - extract outlier frames
        - refine labels using the napari GUI
        - merge datasets
        - train and test

In [ ]:
# import necessary libraries
from ipywidgets import FileUpload, Text, Output
from IPython.display import display
import os
import deeplabcut
%gui qt6
import napari

 

# below is a specific script -- remove before making the general notebook

# the videos folder has videos for both training and testing
# we can identify the labeled videos based on the labeled-data directory
video_fns = os.listdir("C:\\Users\\johnp\\OneDrive\\Desktop\\smellovision\\visuo_olfactory_tracking_w_bar_exp\\dlc\\super_fly_videos\\SuperFly-Pablo-2025-07-29\\videos")
video_fns = [fn for fn in video_fns if fn.endswith('.mp4') and 'labeled' not in fn]
# now, let's move each video_fn to different lists for training and testing based on whether they're included in the labeled-data directory
train_fns, test_fns = [], []
labeled_fns = os.listdir("SuperFly-Pablo-2025-07-29\\labeled-data")
for video_fn in video_fns:
    label_fn = video_fn.split(".")[0] + '_labeled'
    if label_fn in labeled_fns:
        train_fns += [video_fn]
    else:
        test_fns += [video_fn]
all_fns = train_fns + test_fns
train_fns, test_fns

Loading DLC 3.0.0rc11...


(['20250305_1120_fly_1_Drum_gain_30_led_0_rep_1_seq_1_vid.mp4',
  '20250305_1124_fly_2_Drum_gain_-30_led_0_rep_1_seq_1_vid.mp4',
  '20250305_1124_fly_2_Drum_gain_30_led_0_rep_1_seq_6_vid.mp4',
  '20250613_fly_1_rep_1_trial_1_OptoState_OFF_Cond_1_barCW_grndCCW.mp4',
  '20250613_fly_1_rep_1_trial_5_OptoState_ON_Cond_2_barCW_grndSTC.mp4',
  '20250619_fly_3_rep_1_trial_1_OptoState_ON_Cond_2_backgrndCCW.mp4',
  '2025_03_04_11_43_47.mp4',
  '2025_03_18_10_57_00.mp4',
  '2025_04_15_15_09_46.mp4',
  'bad_1_fly_1_trial_1_cond3.mp4',
  'bad_2_fly_4_trial_1_cond1.mp4',
  'bad_3_fly_6_trial_1_cond2.mp4',
  'baja_bad_fly_51_trial_1_cond1_vidData.mp4',
  'baja_bad_fly_51_trial_1_cond2_vidData.mp4',
  'baja_bad_fly_51_trial_3_cond2_vidData.mp4',
  'baja_fly_50_trial_2_cond2_vidData.mp4',
  'baja_fly_59_trial_1_cond2_vidData.mp4',
  'baja_fly_6_trial_1_cond2_vidData.mp4',
  'dani_fly_2_trial_3_cond1_vidData.mp4',
  'dani_fly_5_trial_2_cond2_vidData.mp4',
  'dani_fly_7_trial_1_cond3_vidData.mp4',
  'fl

In [ ]:
# add videos to the video list
fns = os.listdir()
baja_fns = [fn for fn in fns if 'baja_med' in fn]
# use the absolute paths
fns_to_add = [os.path.abspath(fn) for fn in baja_fns]
deeplabcut.add_new_videos(config.path, fns_to_add, copy_videos=True, extract_frames=True)

Config file read successfully.
Frame extraction failed. Video files must be corrupted.
New videos were added to the project and frames have been extracted for labeling!


In [6]:
fns = os.listdir()
baja_fns = [fn for fn in fns if 'baja_bad' in fn]
# use the absolute paths
fns_to_add = [os.path.abspath(fn) for fn in baja_fns]
fns_to_add


['c:\\Users\\johnp\\OneDrive\\Desktop\\smellovision\\visuo_olfactory_tracking_w_bar_exp\\dlc\\super_fly_videos\\baja_bad_fly_51_trial_1_cond1_vidData.mp4',
 'c:\\Users\\johnp\\OneDrive\\Desktop\\smellovision\\visuo_olfactory_tracking_w_bar_exp\\dlc\\super_fly_videos\\baja_bad_fly_51_trial_1_cond2_vidData.mp4',
 'c:\\Users\\johnp\\OneDrive\\Desktop\\smellovision\\visuo_olfactory_tracking_w_bar_exp\\dlc\\super_fly_videos\\baja_bad_fly_51_trial_3_cond2_vidData.mp4',
 'c:\\Users\\johnp\\OneDrive\\Desktop\\smellovision\\visuo_olfactory_tracking_w_bar_exp\\dlc\\super_fly_videos\\baja_bad_fly_52_trial_1_cond1_vidData.mp4',
 'c:\\Users\\johnp\\OneDrive\\Desktop\\smellovision\\visuo_olfactory_tracking_w_bar_exp\\dlc\\super_fly_videos\\baja_bad_fly_52_trial_1_cond2_vidData.mp4']

# Select the videos for training and testing:

In [ ]:
training_videos = FileUpload(accept=('.mp4,.avi,.mat,.mpeg,.h264'), multiple=True)
display(training_videos)

test_videos = FileUpload(accept=('.mp4,.avi,.mat,.mpeg,.h264'), multiple=True)
display(test_videos)

train_fns = [fn['name'] for fn in training_videos.value]
test_fns = [fn['name'] for fn in test_videos.value]

# Create or Load a .yaml config file pointing to all of the videos of interest

In [2]:
# Projects are specified by the task and experimenter names and the date of starting the project.
task = Text(
    value='SuperFly',
    placeholder='Type something',
    description='Task: ',
    disabled=False
)
experimenter = Text(
    value='Pablo',
    placeholder='Type something',
    description='Experimenter: ',
    disabled=False
)
display(task)
display(experimenter)

class Config:
    """Class to handle the configuration for DeepLabCut project creation."""
    def __init__(self):
        self.path = None

    def create_project(self, task, experimenter):
        """
        Create a new DeepLabCut project.

        Parameters
        ----------
        self : Config
            The instance of the Config class.

        Returns
        -------
        str
            The path to the created DeepLabCut project configuration file.
        """
        self.path = deeplabcut.create_new_project(
            task.value,
            experimenter.value,
            train_fns,
            copy_videos=True
        )
        print(f"Project created at: {self.path}")
        return self.path
    
    def load_project(self, task, experimenter):
        """Load an existing DeepLabCut project configuration file.

        Using the provided task and experimenter names, this method finds the most
        recent project configuration file in the current directory and sets the path.

        Parameters
        ----------
        self : Config
            The instance of the Config class.
        task : str
            The name of the task associated with the project.
        experimenter : str
            The name of the experimenter associated with the project.
        
        Returns
        -------
        str
            The path to the loaded DeepLabCut project configuration file.
        """
        # using the task and experimenter names, we can search for subdirectories
        # that match the pattern "task-experimenter-{date}"
        template = f"{task.value}-{experimenter.value}-"
        print(template)
        matching_dirs = [d for d in os.listdir('.') if os.path.isdir(d) and d.startswith(template)]
        # use the most recently created directory
        if matching_dirs:
            most_recent_dir = max(matching_dirs, key=os.path.getctime)
            self.path = os.path.join(most_recent_dir, 'config.yaml')
            print(f"Project loaded from: {self.path}")
        else:
            print("No matching project found.")

# set up the configuration
config = Config()
""" task.value = 'SuperFly'
experimenter.value = 'Pablo'
config.load_project(task, experimenter)
 """
# make a button to create the project using the input from the text boxes
from ipywidgets import Button
create_project_button = Button(description="New", button_style='success')
def on_create_project_button_click(b):
    config.create_project(task, experimenter)
create_project_button.on_click(on_create_project_button_click)

# and another to handle loading the project
load_project_button = Button(description="Load", button_style='info')
def on_load_project_button_click(b):
    config.load_project(task, experimenter)
load_project_button.on_click(on_load_project_button_click)

display(create_project_button)
display(load_project_button)

# keep track of printed values using Output widget
output = Output()
display(output)

Text(value='SuperFly', description='Task: ', placeholder='Type something')

Text(value='Pablo', description='Experimenter: ', placeholder='Type something')

Button(button_style='success', description='New', style=ButtonStyle())

Button(button_style='info', description='Load', style=ButtonStyle())

Output()

# initial training
- extract frames
- label using the napari GUI
- check the labels
- train and test:
    - create training dataset
    - train network
    - evaluate network
    - analyze and automatically label unlabeled videos

In [8]:
# extract frames from the videos and label them
baja_fns = [os.path.join("SuperFly-Pablo-2025-07-29", "videos", fn) for fn in all_fns if 'baja_bad' in fn]
baja_fns = [os.path.abspath(fn) for fn in baja_fns]
deeplabcut.extract_frames(config.path, videos_list=baja_fns, userfeedback=False, mode='automatic')

# napari will pop up!
# Please go to plugin > deeplabcut to start
# then, drag-and-drop the project configuration file into the viewer (the value of path_config_file)
# finally, drop the folder containing the images (in 'labeled-data') in the viewer
napari.Viewer()

Config file read successfully.
Frame extraction failed. Video files must be corrupted.


Viewer(camera=Camera(center=(0.0, 0.0, 0.0), zoom=1.0, angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=True, mouse_zoom=True), cursor=Cursor(position=(1.0, 1.0), scaled=True, size=1, style=<CursorStyle.STANDARD: 'standard'>), dims=Dims(ndim=2, ndisplay=2, last_used=0, range=((0, 2, 1), (0, 2, 1)), current_step=(0, 0), order=(0, 1), axis_labels=('0', '1')), grid=GridCanvas(stride=1, shape=(-1, -1), enabled=False), layers=[], help='', status='Ready', tooltip=Tooltip(visible=False, text=''), theme='dark', title='napari', mouse_over_canvas=False, mouse_move_callbacks=[], mouse_drag_callbacks=[], mouse_double_click_callbacks=[], mouse_wheel_callbacks=[<function dims_scroll at 0x000002166A069EE0>], _persisted_mouse_event={}, _mouse_drag_gen={}, _mouse_wheel_gen={}, keymap={})

In [23]:
baja_fns

['c:\\Users\\johnp\\OneDrive\\Desktop\\smellovision\\visuo_olfactory_tracking_w_bar_exp\\dlc\\super_fly_videos\\SuperFly-Pablo-2025-07-29\\videos\\baja_fly_17_trial_1_cond1_vidData.mp4',
 'c:\\Users\\johnp\\OneDrive\\Desktop\\smellovision\\visuo_olfactory_tracking_w_bar_exp\\dlc\\super_fly_videos\\SuperFly-Pablo-2025-07-29\\videos\\baja_fly_50_trial_2_cond2_vidData.mp4',
 'c:\\Users\\johnp\\OneDrive\\Desktop\\smellovision\\visuo_olfactory_tracking_w_bar_exp\\dlc\\super_fly_videos\\SuperFly-Pablo-2025-07-29\\videos\\baja_fly_6_trial_1_cond2_vidData.mp4']

In [9]:
napari.Viewer()

Viewer(camera=Camera(center=(0.0, 0.0, 0.0), zoom=1.0, angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=True, mouse_zoom=True), cursor=Cursor(position=(1.0, 1.0), scaled=True, size=1, style=<CursorStyle.STANDARD: 'standard'>), dims=Dims(ndim=2, ndisplay=2, last_used=0, range=((0, 2, 1), (0, 2, 1)), current_step=(0, 0), order=(0, 1), axis_labels=('0', '1')), grid=GridCanvas(stride=1, shape=(-1, -1), enabled=False), layers=[], help='', status='Ready', tooltip=Tooltip(visible=False, text=''), theme='dark', title='napari', mouse_over_canvas=False, mouse_move_callbacks=[], mouse_drag_callbacks=[], mouse_double_click_callbacks=[], mouse_wheel_callbacks=[<function dims_scroll at 0x000002166A069EE0>], _persisted_mouse_event={}, _mouse_drag_gen={}, _mouse_wheel_gen={}, keymap={})

In [ ]:
# check the labels by plotting the labeled data in a copied folder
deeplabcut.check_labels(config.path, )

Creating images with labels by Pablo.


100%|██████████| 70/70 [00:02<00:00, 24.50it/s]


Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250305_1129_fly_2_Drum_gain_-30_led_0_rep_2_seq_1_vid does not appear to have labeled data!
Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250305_1129_fly_2_Drum_gain_30_led_0_rep_2_seq_6_vid does not appear to have labeled data!


100%|██████████| 60/60 [00:01<00:00, 37.77it/s]


Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250620_fly_4_rep_1_trial_3_OptoState_OFF_Cond_3_barCW_grndCCW does not appear to have labeled data!
Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250623_fly_6_rep_1_trial_2_OptoState_OFF_Cond_3_barCCW_grndSTC does not appear to have labeled data!


...

Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\good_4_fly_12_trial_1_cond1 does not appear to have labeled data!
Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\good_5_fly_12_trial_1_cond2 does not appear to have labeled data!
Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\dani_fly_10_trial_1_cond2_vidData does not appear to have labeled data!


100%|██████████| 20/20 [00:00<00:00, 37.56it/s]


Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\dani_fly_8_trial_3_cond3_vidData does not appear to have labeled data!
Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\baja_fly_17_trial_1_cond1_vidData does not appear to have labeled data!
Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\baja_fly_46_trial_2_cond2_vidData does not appear to have labeled data!


100%|██████████| 20/20 [00:00<00:00, 23.63it/s]

Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\baja_bad_fly_52_trial_1_cond1_vidData does not appear to have labeled data!
Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\baja_bad_fly_52_trial_1_cond2_vidData does not appear to have labeled data!
If all the labels are ok, then use the function 'create_training_dataset' to create the training dataset!


In [11]:
import numpy as np

def train_and_test():
    """Run a typical training and testing workflow for DeepLabCut."""
    # 1. Create a training dataset
    deeplabcut.create_training_dataset(config.path)
    # 2. Train the network
    deeplabcut.train_network(config.path)
    # 3. Evaluate the network
    deeplabcut.evaluate_network(config.path, plotting=True)
    # 4. Analyze all of the videos
    ftypes = np.array(['.' + fn.split('.')[-1] for fn in all_fns])
    ftypes_set = set(ftypes)
    # run the analysis for each file based on its type
    for ftype in ftypes_set:
        fns = np.array(all_fns)[ftypes == ftype]
        # analyze the videos
        deeplabcut.analyze_videos(config.path, fns.tolist(), videotype=ftype)
    # 5. Create labeled data
    # deeplabcut.create_labeled_video(config.path, all_fns)

In [12]:
train_and_test()

C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250305_1129_fly_2_Drum_gain_-30_led_0_rep_2_seq_1_vid\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250305_1129_fly_2_Drum_gain_30_led_0_rep_2_seq_6_vid\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250620_fly_4_rep_1_trial_3_OptoState_OFF_Cond_3_barCW_grndCCW\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250623_fly_6_rep_1_trial_2_OptoState_OFF_Cond_3_barCCW_grndSTC\CollectedData_Pablo.h5 

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [0.5, 1.25]
      translation: 0
    crop_sampling:
      width: 448
      height: 448
      max_shift: 0.1
      method: hybrid
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
device: auto
metadata:
  project_path: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29
  pose_config_path: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\dlc-models-pytorch\iteration-6\SuperFlyJul29-trainset95shuffle1\train\pytorch_config.yaml
  bodyparts: ['neck', 'tail_joint', 'tail_end', 'antenna_left', 'antenna_right', 'head_left', 'head_right', 'shoulder_left', 'shoulder_right']
  unique_bodyparts: []
  individuals: ['animal']
  with_identity: None
m

FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'SuperFly-Pablo-2025-07-29\\dlc-models-pytorch\\iteration-6\\SuperFlyJul29-trainset95shuffle1\\train\\snapshot-best-100.pt' -> 'SuperFly-Pablo-2025-07-29\\dlc-models-pytorch\\iteration-6\\SuperFlyJul29-trainset95shuffle1\\train\\snapshot-100.pt'

In [17]:
import numpy as np
ftypes = np.array(['.' + fn.split('.')[-1] for fn in all_fns])
ftypes_set = set(ftypes)
ftype = ftypes_set.pop()
fns = np.array(all_fns)[ftypes == ftype]
# analyze the videos
deeplabcut.analyze_videos(config.path, fns.tolist(), videotype=ftype)


Successfully loaded snapshot: SuperFly-Pablo-2025-07-29\dlc-models-pytorch\iteration-4\SuperFlyJul29-trainset95shuffle1\train\snapshot-best-040.pt
Analyzing videos with SuperFly-Pablo-2025-07-29\dlc-models-pytorch\iteration-4\SuperFlyJul29-trainset95shuffle1\train\snapshot-best-040.pt
DEBUG: No prefix mismatch, loading original weights
Using scorer: DLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-40
Starting to analyze 20250305_1120_fly_1_Drum_gain_30_led_0_rep_1_seq_1_vid.mp4
Video metadata: 
  Overall # of frames:    3963
  Duration of video [s]:  132.10
  fps:                    30.0
  resolution:             w=420, h=420

Running pose prediction with batch size 8


Pose: 100%|██████████| 3963/3963 [01:22<00:00, 47.82it/s]


Saving results in 20250305_1120_fly_1_Drum_gain_30_led_0_rep_1_seq_1_vidDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-40.h5 and 20250305_1120_fly_1_Drum_gain_30_led_0_rep_1_seq_1_vidDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-40_full.pickle
Starting to analyze 20250305_1124_fly_2_Drum_gain_-30_led_0_rep_1_seq_1_vid.mp4
Video metadata: 
  Overall # of frames:    5000
  Duration of video [s]:  166.67
  fps:                    30.0
  resolution:             w=420, h=420

Running pose prediction with batch size 8


Pose: 100%|██████████| 5000/5000 [01:45<00:00, 47.30it/s]


Saving results in 20250305_1124_fly_2_Drum_gain_-30_led_0_rep_1_seq_1_vidDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-40.h5 and 20250305_1124_fly_2_Drum_gain_-30_led_0_rep_1_seq_1_vidDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-40_full.pickle
Starting to analyze 20250305_1124_fly_2_Drum_gain_30_led_0_rep_1_seq_6_vid.mp4
Video metadata: 
  Overall # of frames:    5000
  Duration of video [s]:  166.67
  fps:                    30.0
  resolution:             w=420, h=420

Running pose prediction with batch size 8


...

Saving results in 20250613_fly_1_rep_1_trial_5_OptoState_ON_Cond_2_barCW_grndSTCDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-40.h5 and 20250613_fly_1_rep_1_trial_5_OptoState_ON_Cond_2_barCW_grndSTCDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-40_full.pickle
Starting to analyze 20250619_fly_3_rep_1_trial_1_OptoState_ON_Cond_2_backgrndCCW.mp4
Video metadata: 
  Overall # of frames:    1003
  Duration of video [s]:  10.03
  fps:                    100.0
  resolution:             w=420, h=420

Running pose prediction with batch size 8


Pose: 100%|██████████| 1003/1003 [00:21<00:00, 46.51it/s]


Saving results in 20250619_fly_3_rep_1_trial_1_OptoState_ON_Cond_2_backgrndCCWDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-40.h5 and 20250619_fly_3_rep_1_trial_1_OptoState_ON_Cond_2_backgrndCCWDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-40_full.pickle
Starting to analyze 2025_03_04_11_43_47.mp4
Video metadata: 
  Overall # of frames:    18086
  Duration of video [s]:  90.43
  fps:                    200.0
  resolution:             w=720, h=540

Running pose prediction with batch size 8


Pose:  60%|█████▉    | 10848/18086 [04:58<21:56,  5.50it/s]

: 

In [14]:
# train and test the network
train_and_test()

C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250305_1129_fly_2_Drum_gain_-30_led_0_rep_2_seq_1_vid\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250305_1129_fly_2_Drum_gain_30_led_0_rep_2_seq_6_vid\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250620_fly_4_rep_1_trial_3_OptoState_OFF_Cond_3_barCW_grndCCW\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250623_fly_6_rep_1_trial_2_OptoState_OFF_Cond_3_barCCW_grndSTC\CollectedData_Pablo.h5 

Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\dani_fly_8_trial_3_cond3_vidData\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\baja_fly_17_trial_1_cond1_vidData\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\baja_fly_46_trial_2_cond2_vidData\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\baja_bad_fly_52_trial_1_cond1_vidData\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_trac

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [0.5, 1.25]
      translation: 0
    crop_sampling:
      width: 448
      height: 448
      max_shift: 0.1
      method: hybrid
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
device: auto
metadata:
  project_path: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29
  pose_config_path: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\dlc-models-pytorch\iteration-7\SuperFlyJul29-trainset95shuffle1\train\pytorch_config.yaml
  bodyparts: ['neck', 'tail_joint', 'tail_end', 'antenna_left', 'antenna_right', 'head_left', 'head_right', 'shoulder_left', 'shoulder_right']
  unique_bodyparts: []
  individuals: ['animal']
  with_identity: None
m

Evaluation scorer: DLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-100
DEBUG: No prefix mismatch, loading original weights


...

Saving results in 20250613_fly_1_rep_1_trial_5_OptoState_ON_Cond_2_barCW_grndSTCDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-100.h5 and 20250613_fly_1_rep_1_trial_5_OptoState_ON_Cond_2_barCW_grndSTCDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-100_full.pickle
Starting to analyze 20250619_fly_3_rep_1_trial_1_OptoState_ON_Cond_2_backgrndCCW.mp4
Video metadata: 
  Overall # of frames:    1003
  Duration of video [s]:  10.03
  fps:                    100.0
  resolution:             w=420, h=420

Running pose prediction with batch size 8


Pose: 100%|██████████| 1003/1003 [00:21<00:00, 45.71it/s]


Saving results in 20250619_fly_3_rep_1_trial_1_OptoState_ON_Cond_2_backgrndCCWDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-100.h5 and 20250619_fly_3_rep_1_trial_1_OptoState_ON_Cond_2_backgrndCCWDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-100_full.pickle
Starting to analyze 2025_03_04_11_43_47.mp4
Video metadata: 
  Overall # of frames:    18086
  Duration of video [s]:  90.43
  fps:                    200.0
  resolution:             w=720, h=540

Running pose prediction with batch size 8


Pose:  55%|█████▌    | 10010/18086 [04:37<14:24,  9.35it/s]

: 

# refining
- extract outlier frames
- refine labels using the napari GUI
- check labels and merge datasets
- train and test

In [4]:
# extract outlier frames from the training dataset
deeplabcut.extract_outlier_frames(
    config.path,
    train_fns,
)

Method  jump  found  3108  putative outlier frames.
Do you want to proceed with extracting  20  of those?
If this list is very large, perhaps consider changing the parameters (start, stop, p_bound, comparisonbodyparts) or use a different method.


c:\Users\johnp\.conda\envs\DEEPLABCUT\Lib\site-packages\deeplabcut\refine_training_dataset\outlier_frames.py:418: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  sum_ = temp_dt.groupby(level="bodyparts", axis=1).sum()


Frames from video 20250305_1120_fly_1_Drum_gain_30_led_0_rep_1_seq_1_vid  already extracted (more will be added)!
Loading video...
Cropping coords: None
Duration of video [s]:  132.1 , recorded @  30.0 fps!
Overall # of frames:  3963 with (cropped) frame dimensions: 
Kmeans-quantization based extracting of frames from 0.0  seconds to 132.1  seconds.
Extracting and downsampling... 3108  frames from the video.


3108it [01:26, 35.75it/s] 
c:\Users\johnp\.conda\envs\DEEPLABCUT\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 8192 or by setting the environment variable OMP_NUM_THREADS=1
  warnings.warn(


Kmeans clustering ... (this might take a while)
Let's select frames indices: [1194, 2186, 2061, 403, 2378, 319, 3058, 2002, 2861, 1724, 3658, 36, 2105, 245, 2735, 909, 259, 1498, 3196, 2082]
Attempting to create a symbolic link of the video ...
Video SuperFly-Pablo-2025-07-29\videos\20250305_1120_fly_1_Drum_gain_30_led_0_rep_1_seq_1_vid.mp4 already exists. Skipping...
New videos were added to the project! Use the function 'extract_frames' to select frames for labeling.
The outlier frames are extracted. They are stored in the subdirectory labeled-data\20250305_1120_fly_1_Drum_gain_30_led_0_rep_1_seq_1_vid.
Once you extracted frames for all videos, use 'refine_labels' to manually correct the labels.
Method  jump  found  10  putative outlier frames.
Do you want to proceed with extracting  20  of those?
If this list is very large, perhaps consider changing the parameters (start, stop, p_bound, comparisonbodyparts) or use a different method.


...

Let's select frames indices: [842, 803, 844, 1042, 1047, 391, 1044, 1041, 806, 801, 800, 1046, 802, 1048, 771, 843, 841]
Attempting to create a symbolic link of the video ...
Video SuperFly-Pablo-2025-07-29\videos\good_1_fly_7_trial_1_cond1.mp4 already exists. Skipping...
New videos were added to the project! Use the function 'extract_frames' to select frames for labeling.
The outlier frames are extracted. They are stored in the subdirectory labeled-data\good_1_fly_7_trial_1_cond1.
Once you extracted frames for all videos, use 'refine_labels' to manually correct the labels.
Method  jump  found  0  putative outlier frames.
Do you want to proceed with extracting  20  of those?
If this list is very large, perhaps consider changing the parameters (start, stop, p_bound, comparisonbodyparts) or use a different method.


c:\Users\johnp\.conda\envs\DEEPLABCUT\Lib\site-packages\deeplabcut\refine_training_dataset\outlier_frames.py:418: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  sum_ = temp_dt.groupby(level="bodyparts", axis=1).sum()


Frames from video good_2_fly_10_trial_2_cond1  already extracted (more will be added)!
Loading video...
Cropping coords: None
Duration of video [s]:  60.0 , recorded @  25.0 fps!
Overall # of frames:  1500 with (cropped) frame dimensions: 
Kmeans-quantization based extracting of frames from 0.0  seconds to 60.0  seconds.
Let's select frames indices: []
No frames were extracted.
Method  jump  found  19  putative outlier frames.
Do you want to proceed with extracting  20  of those?
If this list is very large, perhaps consider changing the parameters (start, stop, p_bound, comparisonbodyparts) or use a different method.


c:\Users\johnp\.conda\envs\DEEPLABCUT\Lib\site-packages\deeplabcut\refine_training_dataset\outlier_frames.py:418: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  sum_ = temp_dt.groupby(level="bodyparts", axis=1).sum()


Frames from video good_3_fly_11_trial_1_cond1  already extracted (more will be added)!
Loading video...
Cropping coords: None
Duration of video [s]:  60.0 , recorded @  25.0 fps!
Overall # of frames:  1500 with (cropped) frame dimensions: 
Kmeans-quantization based extracting of frames from 0.0  seconds to 60.0  seconds.
Let's select frames indices: [211, 212, 700, 701, 702, 703, 705, 706, 707, 708, 709, 948, 949, 950, 955, 956, 1171, 1177, 1178]
Attempting to create a symbolic link of the video ...
Video SuperFly-Pablo-2025-07-29\videos\good_3_fly_11_trial_1_cond1.mp4 already exists. Skipping...
New videos were added to the project! Use the function 'extract_frames' to select frames for labeling.
The outlier frames are extracted. They are stored in the subdirectory labeled-data\good_3_fly_11_trial_1_cond1.
Once you extracted frames for all videos, use 'refine_labels' to manually correct the labels.


In [11]:
# refine labels for the training dataset using the napari GUI
deeplabcut.refine_labels(config.path)

In [8]:
# check the new labels and merge the datasets
deeplabcut.check_labels(config.path)
# make sure that all of the training videos in the labeled-data directory are properly labeled

Creating images with labels by Pablo.


100%|██████████| 70/70 [00:02<00:00, 23.97it/s]


Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250305_1129_fly_2_Drum_gain_-30_led_0_rep_2_seq_1_vid does not appear to have labeled data!
Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250305_1129_fly_2_Drum_gain_30_led_0_rep_2_seq_6_vid does not appear to have labeled data!


100%|██████████| 60/60 [00:01<00:00, 36.53it/s]


Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250620_fly_4_rep_1_trial_3_OptoState_OFF_Cond_3_barCW_grndCCW does not appear to have labeled data!
Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250623_fly_6_rep_1_trial_2_OptoState_OFF_Cond_3_barCCW_grndSTC does not appear to have labeled data!


...

100%|██████████| 20/20 [00:00<00:00, 102.67it/s]


Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\good_4_fly_12_trial_1_cond1 does not appear to have labeled data!
Attention: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\good_5_fly_12_trial_1_cond2 does not appear to have labeled data!


100%|██████████| 20/20 [00:00<00:00, 104.07it/s]

If all the labels are ok, then use the function 'create_training_dataset' to create the training dataset!


c:\Users\johnp\.conda\envs\DEEPLABCUT\Lib\site-packages\napari_deeplabcut\_widgets.py:529: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  self.ax.set_xlim(start, end)


In [13]:
# if so, then the following should work
deeplabcut.merge_datasets(config.path)
# deeplabcut.merge_datasets?


Merged data sets and updated refinement iteration to 7.
Now you can create a new training set for the expanded annotated images (use create_training_dataset).


In [ ]:
train_and_test()

C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250305_1129_fly_2_Drum_gain_-30_led_0_rep_2_seq_1_vid\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250305_1129_fly_2_Drum_gain_30_led_0_rep_2_seq_6_vid\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250620_fly_4_rep_1_trial_3_OptoState_OFF_Cond_3_barCW_grndCCW\CollectedData_Pablo.h5  not found (perhaps not annotated).
C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\labeled-data\20250623_fly_6_rep_1_trial_2_OptoState_OFF_Cond_3_barCCW_grndSTC\CollectedData_Pablo.h5 

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [0.5, 1.25]
      translation: 0
    crop_sampling:
      width: 448
      height: 448
      max_shift: 0.1
      method: hybrid
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
device: auto
metadata:
  project_path: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29
  pose_config_path: C:\Users\johnp\OneDrive\Desktop\smellovision\visuo_olfactory_tracking_w_bar_exp\dlc\super_fly_videos\SuperFly-Pablo-2025-07-29\dlc-models-pytorch\iteration-5\SuperFlyJul29-trainset95shuffle1\train\pytorch_config.yaml
  bodyparts: ['neck', 'tail_joint', 'tail_end', 'antenna_left', 'antenna_right', 'head_left', 'head_right', 'shoulder_left', 'shoulder_right']
  unique_bodyparts: []
  individuals: ['animal']
  with_identity: None
m

Evaluation scorer: DLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-50
DEBUG: No prefix mismatch, loading original weights


100%|██████████| 49/49 [00:00<00:00, 115.54it/s]


Evaluation results file: DLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-50-results.csv
Evaluation results for DLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-50-results.csv (pcutoff: 0.6):
train rmse              3.16
train rmse_pcutoff      2.62
train mAP              98.37
train mAR              99.19
test rmse               3.22
test rmse_pcutoff       3.00
test mAP              100.00
test mAR              100.00
Name: (0.95, 1, 50, -1, 0.6), dtype: float64
Successfully loaded snapshot: SuperFly-Pablo-2025-07-29\dlc-models-pytorch\iteration-5\SuperFlyJul29-trainset95shuffle1\train\snapshot-best-050.pt
Analyzing videos with SuperFly-Pablo-2025-07-29\dlc-models-pytorch\iteration-5\SuperFlyJul29-trainset95shuffle1\train\snapshot-best-050.pt
DEBUG: No prefix mismatch, loading original weights
Using scorer: DLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-50
Starting to analyze 20250305_1120_fly_1_Drum_gain_30_led_0_rep_1_seq_1_vid.mp4
Video metadata: 
  Overall # of frames:    3963


...

Pose: 100%|██████████| 1021/1021 [00:14<00:00, 69.54it/s]


Saving results in 20250613_fly_1_rep_1_trial_5_OptoState_ON_Cond_2_barCW_grndSTCDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-50.h5 and 20250613_fly_1_rep_1_trial_5_OptoState_ON_Cond_2_barCW_grndSTCDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-50_full.pickle
Starting to analyze 20250619_fly_3_rep_1_trial_1_OptoState_ON_Cond_2_backgrndCCW.mp4
Video metadata: 
  Overall # of frames:    1003
  Duration of video [s]:  10.03
  fps:                    100.0
  resolution:             w=420, h=420

Running pose prediction with batch size 8


Pose: 100%|██████████| 1003/1003 [00:14<00:00, 68.16it/s]


Saving results in 20250619_fly_3_rep_1_trial_1_OptoState_ON_Cond_2_backgrndCCWDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-50.h5 and 20250619_fly_3_rep_1_trial_1_OptoState_ON_Cond_2_backgrndCCWDLC_Resnet50_SuperFlyJul29shuffle1_snapshot_best-50_full.pickle
Starting to analyze 2025_03_04_11_43_47.mp4
Video metadata: 
  Overall # of frames:    18086
  Duration of video [s]:  90.43
  fps:                    200.0
  resolution:             w=720, h=540

Running pose prediction with batch size 8


Pose:  61%|██████    | 11017/18086 [03:48<04:57, 23.78it/s]

In [ ]:
os.path.isdir